## 设置

要完成以下指南，您需要安装以下软件包：
- anthropic 
- voyageai
- pandas
- matplotlib
- sklearn
- numpy

您还需要：

- Anthropic API密钥
- VoyageAI API密钥（可选）
    - 嵌入已预计算，但如果您进行任何更改则需要API密钥
- DeepSeek API密钥（可选）
    - 使用DeepSeek的Anthropic兼容API: `https://api.deepseek.com/anthropic`

In [ ]:
!pip install anthropic
!pip install voyageai
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install -U scikit-learn

In [ ]:
import os

# 设置API密钥
os.environ['VOYAGE_API_KEY'] = "VOYAGE AI的API密钥"
# os.environ['ANTHROPIC_API_KEY'] = "ANTHROPIC密钥在此"

# DeepSeek API配置（可选）
# 使用DeepSeek的Anthropic兼容API
os.environ['DEEPSEEK_API_KEY'] = "DeepSeek的API密钥"
os.environ['ANTHROPIC_API_BASE'] = "https://api.deepseek.com/anthropic/"

In [ ]:
# 设置环境
import anthropic
import os

client = anthropic.Anthropic(
    # 这是默认值，可以省略
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

In [ ]:
# 设置环境
import anthropic
import os

# 标准Anthropic配置
# client = anthropic.Anthropic(
#     # 这是默认值，可以省略
#     api_key=os.getenv("ANTHROPIC_API_KEY"),
# )

# DeepSeek配置（可选）
# 如果您想使用DeepSeek的Anthropic兼容API，请取消下面的注释
# 打印环境变量
print(f"ANTHROPIC_API_BASE: {os.getenv('ANTHROPIC_API_BASE')}")
print(f"DEEPSEEK_API_KEY: {os.getenv('DEEPSEEK_API_KEY')}")
client = anthropic.Anthropic(
    base_url=os.getenv("ANTHROPIC_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
)

## 测试正常使用的例子

下面的代码测试正常使用的例子。

测试发现项目例子有这个问题。

设置了stop_sequences=["</category>"]，希望模型生成到</category>时停止。但由于提前在对话历史中加入了<category>，模型可能会误解为 “需要接着<category>继续生成”，导致生成内容不完整或偏离预期。

```python
def demo():
    prompt = textwrap.dedent("""
    你是谁？
    """)
    print("prompt：", prompt)
    response = client.messages.create( 
        messages=[{"role":"user", "content": prompt}],
        stop_sequences=["</category>"], 
        max_tokens=4096, 
        temperature=0.0,
        model="deepseek-chat"
    )
    
    # 从响应中提取结果
    result = response.content[0].text.strip()
    print("result：", response.content)
    return result

demo()
```

# 使用Claude进行分类

大型语言模型（LLMs）已经彻底改变了分类领域，特别是在传统机器学习系统面临挑战的领域。LLMs在处理具有复杂业务规则和低质量或有限训练数据的场景的分类问题方面表现出显著的成功。此外，LLMs具有为其行为提供自然语言解释和理由的能力，增强了分类过程的可解释性和透明度。通过利用LLMs的强大功能，我们可以构建超越传统机器学习方法能力的分类系统，并在数据稀缺或业务要求复杂的场景中表现出色。

在本指南中，我们将探讨如何利用LLMs来处理高级分类任务。我们将涵盖以下关键组成部分和步骤：

1. **数据准备**：我们将首先准备我们的训练和测试数据。训练数据将用于构建分类模型，而测试数据将用于评估其性能。正确的数据准备对于确保我们分类系统的有效性至关重要。

2. **提示工程**：提示工程在利用LLMs进行分类方面起着至关重要的作用。我们将设计一个提示模板，定义用于分类的提示的结构和格式。提示模板将包含用户查询、类别定义和来自向量数据库的相关示例。通过仔细设计提示，我们可以引导LLM生成准确且与上下文相关的分类。

3. **实施检索增强生成（RAG）**：为了增强分类过程，我们将使用向量数据库来存储和高效检索我们训练数据的嵌入。向量数据库支持相似性搜索，使我们能够为给定查询找到最相关的示例。通过使用检索到的示例增强LLM，我们可以提供额外的上下文并提高生成分类的准确性。

4. **测试和评估**：一旦我们的分类系统构建完成，我们将使用转换后的测试数据严格测试其性能。我们将遍历测试查询，使用分类函数对每个查询进行分类，并将预测的类别与预期类别进行比较。通过分析分类结果，我们可以评估系统的有效性并确定需要改进的领域。

## 问题定义：保险支持票据分类器

*注：本示例中使用的问题定义、数据和标签由Claude 3 Opus合成生成*

在保险行业中，客户支持在确保客户满意度和留存方面发挥着至关重要的作用。保险公司每天收到大量支持票据，涵盖广泛的主题，如账单、保单管理、理赔协助等。手动分类这些票据可能耗时且效率低下，导致响应时间延长并可能影响客户体验。

#### 类别定义

1. 账单查询
- 关于发票、费用、收费和保费的问题
- 要求澄清账单报表
- 关于付款方式和到期日的询问

2. 保单管理
- 要求更改、更新或取消保单
- 关于保单续保和恢复的问题
- 关于添加或删除保险选项的询问

3. 理赔协助
- 关于理赔流程和申请程序的问题
- 要求协助提交理赔文件
- 关于理赔状态和赔付时间的询问

4. 保险范围解释
- 关于特定保单类型涵盖内容的问题
- 要求澄清保险范围限制和除外条款
- 关于免赔额和自付费用的询问


5. 报价和建议
- 要求新的保单报价和价格比较
- 关于可用折扣和捆绑选项的问题
- 关于从其他保险公司转保的询问


6. 账户管理
- 要求登录凭据或密码重置
- 关于在线账户特性和功能的问题
- 关于更新联系信息或个人信息的询问


7. 账单争议
- 关于意外或不正确收费的投诉
- 要求退款或保费调整
- 关于滞纳金或催款通知的询问


8. 理赔争议
- 关于被拒绝或赔付不足的理赔的投诉
- 要求重新考虑理赔决定
- 关于申诉理赔结果的询问


9. 保单比较
- 关于保单选项之间差异的问题
- 要求帮助决定保险等级
- 关于与竞争对手产品比较的询问


10. 一般询问
- 关于公司联系信息或营业时间的问题
- 关于产品或服务的一般信息请求
- 不适合其他类别的询问

#### 标记数据

我们将使用以下数据集：
- `./data/test-less.tsv`
- `./data/train-less.tsv`

In [ ]:
import pandas as pd

data = {
    'train': [],
    'test': [],
    'test_2': []
}

# Helper function to convert a DataFrame to a list of dictionaries
def dataframe_to_dict_list(df):
    return df.apply(lambda x: {'text': x['text'], 'label': x['label']}, axis=1).tolist()


# Read the TSV file into a DataFrame
test_df = pd.read_csv("./data/test-less.tsv", sep='\t')
data['test'] = dataframe_to_dict_list(test_df)

train_df = pd.read_csv("./data/train-less.tsv", sep='\t')
data['train'] = dataframe_to_dict_list(train_df)


# Understand the labels in the dataset
labels = list(set(train_df['label'].unique()))

# Print the first training example and the number of training examples
print(data['train'][0], len(data['train']))

# Create the test set
X_test = [example['text'] for example in data['test']]
y_test = [example['label'] for example in data['test']]

# Print the length of the test set
print(len(X_test), len(y_test))

### 评估每个分类模型

`evaluate`函数接受以下参数：
- `X`：输入特征。
- `y`：真实标签。
- `classifier`：要评估的分类器函数。
- `batch_size`：分类的每批大小（默认为层级的最大批大小）。

`plot_confusion_matrix`函数接受以下参数：
- `cm`：混淆矩阵。
- `labels`：类别的标签。

通过使用此评估代码，您可以评估分类器的性能并可视化混淆矩阵，以深入了解模型的预测。

调整`MAXIMUM_CONCURRENT_REQUESTS`以匹配您的Anthropic账户相关的速率限制，[参见此处](https://docs.anthropic.com/claude/reference/rate-limits)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import concurrent.futures
import numpy as np

# 您可以增加此数字以加快评估速度，但请记住您可能需要更高的API速率限制
# 有关详细信息，请参见 https://docs.anthropic.com/en/api/rate-limits#rate-limits
MAXIMUM_CONCURRENT_REQUESTS = 5

def plot_confusion_matrix(cm, labels):
    # 可视化混淆矩阵
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(cm, cmap='Blues')

    # 添加颜色条
    cbar = ax.figure.colorbar(im, ax=ax)

    # 设置刻度标签和位置
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)

    # 为每个单元格添加标签
    thresh = cm.max() / 2.
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j],
                    ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black')

    # 设置标签和标题
    plt.xlabel('预测标签')
    plt.ylabel('真实标签')
    plt.title('混淆矩阵')
    plt.tight_layout()
    plt.show()

def evaluate(X, y, classifier, batch_size=MAXIMUM_CONCURRENT_REQUESTS):
    # 初始化列表以存储预测和真实标签
    y_true = []
    y_pred = []

    # 创建一个ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # 将分类任务分批提交给执行器
        futures = []
        for i in range(0, len(X), batch_size):
            batch_X = X[i:i+batch_size]
            batch_futures = [executor.submit(classifier, x) for x in batch_X]
            futures.extend(batch_futures)

        # 按原始顺序检索结果
        for i, future in enumerate(futures):
            # 打印大模型的返回
            predicted_label = future.result()
            print("大模型返回内容  predicted_label：", predicted_label)
            y_pred.append(predicted_label)
            y_true.append(y[i])

    # 规范化y_true和y_pred
    y_true = [label.strip() for label in y_true]
    y_pred = [label.strip() for label in y_pred]

    # 计算分类指标
    report = classification_report(y_true, y_pred, labels=labels, zero_division=1)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(report)
    plot_confusion_matrix(cm, labels)

### 评估每个分类模型

`evaluate`函数接受以下参数：
- `X`：输入特征。
- `y`：真实标签。
- `classifier`：要评估的分类器函数。
- `batch_size`：分类的每批大小（默认为层级的最大批大小）。

`plot_confusion_matrix`函数接受以下参数：
- `cm`：混淆矩阵。
- `labels`：类别的标签。

通过使用此评估代码，您可以评估分类器的性能并可视化混淆矩阵，以深入了解模型的预测。

调整`MAXIMUM_CONCURRENT_REQUESTS`以匹配您的Anthropic账户相关的速率限制，[参见此处](https://docs.anthropic.com/claude/reference/rate-limits)

### 随机分类器

为了演示我们的evaluate函数的输出，我们可以定义一个随机分类器。

In [ ]:
import random

def random_classifier(text):
    return random.choice(labels)


In [ ]:
print("Evaluating the random classification method on the test set...")
evaluate(X_test, y_test, random_classifier)

### 简单分类测试

现在让我们使用Claude构建一个简单的分类器。

首先，我们将以XML格式编码类别。这将使Claude更容易解释信息。以XML格式编码信息是一种通用的提示策略，更多信息请[参见此处](https://docs.anthropic.com/claude/docs/use-xml-tags)。

In [ ]:
import textwrap
categories = textwrap.dedent("""<category> 
    <label>Billing Inquiries</label>
    <content> Questions about invoices, charges, fees, and premiums Requests for clarification on billing statements Inquiries about payment methods and due dates 
    </content> 
</category> 
<category> 
    <label>Policy Administration</label>
    <content> Requests for policy changes, updates, or cancellations Questions about policy renewals and reinstatements Inquiries about adding or removing coverage options 
    </content> 
</category> 
<category> 
    <label>Claims Assistance</label> 
    <content> Questions about the claims process and filing procedures Requests for help with submitting claim documentation Inquiries about claim status and payout timelines 
    </content> 
</category> 
<category> 
    <label>Coverage Explanations</label> 
    <content> Questions about what is covered under specific policy types Requests for clarification on coverage limits and exclusions Inquiries about deductibles and out-of-pocket expenses 
    </content> 
</category> 
<category> 
    <label>Quotes and Proposals</label> 
    <content> Requests for new policy quotes and price comparisons Questions about available discounts and bundling options Inquiries about switching from another insurer 
    </content> 
</category> 
<category> 
    <label>Account Management</label> 
    <content> Requests for login credentials or password resets Questions about online account features and functionality Inquiries about updating contact or personal information 
    </content> 
</category> 
<category> 
    <label>Billing Disputes</label> 
    <content> Complaints about unexpected or incorrect charges Requests for refunds or premium adjustments Inquiries about late fees or collection notices 
    </content> 
</category> 
<category> 
    <label>Claims Disputes</label> 
    <content> Complaints about denied or underpaid claims Requests for reconsideration of claim decisions Inquiries about appealing a claim outcome 
    </content> 
</category> 
<category> 
    <label>Policy Comparisons</label> 
    <content> Questions about the differences between policy options Requests for help deciding between coverage levels Inquiries about how policies compare to competitors' offerings 
    </content> 
</category> 
<category> 
    <label>General Inquiries</label> 
    <content> Questions about company contact information or hours of operation Requests for general information about products or services Inquiries that don't fit neatly into other categories 
    </content> 
</category>""")

接下来我们将构建一个分类函数，它执行以下操作：
- 定义提示模板
- 在提示模板中输入变量
- 提取标准化的响应

请注意，我们利用`role: assistant`消息和`stop_sequences`参数来可重复地提取结果。

**deepseek测试无法完成。通过函数处理多余的字符串。需要删除多余的提示词。**

In [ ]:
def replace_category(text):
    return text.replace("<category>", "").replace("</category>", "")

In [ ]:
def simple_classify(X):
    prompt = textwrap.dedent("""
    您将把客户支持工单分类到以下类别之一：
    <categories>
        {{categories}}
    </categories>

    以下是客户支持工单：
    <ticket>
        {{ticket}}
    </ticket>

    仅使用类别标签回复，并将其置于category标签之间。
    """).replace("{{categories}}", categories).replace("{{ticket}}", X)
    response = client.messages.create( 
        messages=[{"role":"user", "content": prompt}],
        stop_sequences=["</category>"], 
        max_tokens=4096, 
        temperature=0.0,
        model="deepseek-chat"
    )
    
    # 从响应中提取结果
    result = response.content[0].text.strip()

    return replace_category(result)


In [ ]:
print("正在测试集上评估简单分类方法...")
evaluate(X_test, y_test, simple_classify)

这些结果比随机分类要好，但肯定可以改进！让我们在提示中添加带有K-shot示例的RAG。

为此，我们需要利用向量数据库，这将允许我们将给定查询与训练数据中的相似示例进行匹配。这些示例有望帮助提高分类器的准确性。

我们将构建一个简单的向量数据库类，它利用[VoyageAI](https://docs.anthropic.com/en/docs/embeddings)创建的嵌入模型。

In [ ]:
import os
import numpy as np
import voyageai
import pickle
import json

class VectorDB:
    def __init__(self, api_key=None):
        if api_key is None:
            api_key = os.getenv("VOYAGE_API_KEY")
        self.client = voyageai.Client(api_key=api_key)
        self.embeddings = []
        self.metadata = []
        self.query_cache = {}
        self.db_path = "./data/vector_db.pkl"

    def load_data(self, data):
        # 检查向量数据库是否已经加载
        if self.embeddings and self.metadata:
            print("向量数据库已加载。跳过数据加载。")
            return
        # 检查vector_db.pkl是否存在
        if os.path.exists(self.db_path):
            print("从磁盘加载向量数据库。")
            self.load_db()
            return

        texts = [item["text"] for item in data]

        # 使用for循环嵌入超过128个文档
        batch_size = 128
        result = [
            self.client.embed(
                texts[i : i + batch_size],
                model="voyage-2"
            ).embeddings
            for i in range(0, len(texts), batch_size)
        ]

        # 展平嵌入
        self.embeddings = [embedding for batch in result for embedding in batch]
        self.metadata = [item for item in data]
        self.save_db()
        # 将向量数据库保存到磁盘
        print("向量数据库已加载并保存。")

    def search(self, query, k=5, similarity_threshold=0.75):
        query_embedding = None
        if query in self.query_cache:
            query_embedding = self.query_cache[query]
        else:
            query_embedding = self.client.embed([query], model="voyage-2").embeddings[0]
            self.query_cache[query] = query_embedding

        if not self.embeddings:
            raise ValueError("向量数据库中未加载数据。")

        similarities = np.dot(self.embeddings, query_embedding)
        top_indices = np.argsort(similarities)[::-1]
        top_examples = []
        
        for idx in top_indices:
            if similarities[idx] >= similarity_threshold:
                example = {
                    "metadata": self.metadata[idx],
                    "similarity": similarities[idx],
                }
                top_examples.append(example)
                
                if len(top_examples) >= k:
                    break
        self.save_db()
        return top_examples
    
    def save_db(self):
        data = {
            "embeddings": self.embeddings,
            "metadata": self.metadata,
            "query_cache": json.dumps(self.query_cache),
        }
        with open(self.db_path, "wb") as file:
            pickle.dump(data, file)

    def load_db(self):
        if not os.path.exists(self.db_path):
            raise ValueError("未找到向量数据库文件。使用load_data创建新数据库。")
        
        with open(self.db_path, "rb") as file:
            data = pickle.load(file)
        
        self.embeddings = data["embeddings"]
        self.metadata = data["metadata"]
        self.query_cache = json.loads(data["query_cache"])

我们可以定义向量数据库并加载训练数据。

VoyageAI对于没有关联信用卡的账户有3RPM的速率限制。为了便于演示，我们将利用缓存。

In [ ]:
vectordb = VectorDB()
vectordb.load_data(data["train"])

### RAG分类提示

在这个提示中，我们利用检索增强生成（RAG）来插入来自训练数据的语义上相似查询的示例。

In [ ]:
def rag_classify(X):
    rag = vectordb.search(X,5)
    rag_string = ""
    for example in rag:
        rag_string += textwrap.dedent(f"""
        <example>
            <query>
                "{example["metadata"]["text"]}"
            </query>
            <label>
                {example["metadata"]["label"]}
            </label>
        </example>
        """)
    prompt = textwrap.dedent("""
    您将把客户支持工单分类到以下类别之一：
    <categories>
        {{categories}}
    </categories>

    以下是客户支持工单：
    <ticket>
        {{ticket}}
    </ticket>

    使用以下示例来帮助您对查询进行分类：
    <examples>
        {{examples}}
    </examples>

    仅使用类别标签回复，并将其置于category标签之间。
    """).replace("{{categories}}", categories).replace("{{ticket}}", X).replace("{{examples}}", rag_string)
    response = client.messages.create( 
        messages=[{"role":"user", "content": prompt}],
        stop_sequences=["</category>"], 
        max_tokens=4096, 
        temperature=0.0,
        model="deepseek-chat"
    )
    
    # 从响应中提取结果
    result = response.content[0].text.strip()

    return replace_category(result)

In [ ]:
print("正在测试集上评估RAG方法...")
evaluate(X_test, y_test, rag_classify)

### 使用思维链提示的RAG分类

这个提示将在之前的基础上构建，通过允许Claude反思结果，我们可以提高分类的准确性。

In [ ]:
def rag_chain_of_thought_classify(X):
    rag = vectordb.search(X,5)
    rag_string = ""
    for example in rag:
        rag_string += textwrap.dedent(f"""
        <example>
            <query>
                "{example["metadata"]["text"]}"
            </query>
            <label>
                {example["metadata"]["label"]}
            </label>
        </example>
        """)
    prompt = textwrap.dedent("""
    您将把客户支持工单分类到以下类别之一：
    <categories>
        {{categories}}
    </categories>

    以下是客户支持工单：
    <ticket>
        {{ticket}}
    </ticket>

    使用以下示例来帮助您对查询进行分类：
    <examples>
        {{examples}}
    </examples>

    首先，您将在scratchpad标签中逐步思考问题。
    您应该考虑所有提供的信息，并为您的分类创建一个具体的论据。
    
    使用以下格式回复：
    <response>
        <scratchpad>您的想法和分析放在这里</scratchpad>
        <category>您选择的类别标签放在这里</category>
    </response>
    """).replace("{{categories}}", categories).replace("{{ticket}}", X).replace("{{examples}}", rag_string)
    response = client.messages.create( 
        messages=[{"role":"user", "content": prompt}],
        stop_sequences=["</category>"], 
        max_tokens=4096, 
        temperature=0.0,
        model="deepseek-chat"
    )
    
    # 从响应中提取结果
    result = response.content[0].text.split("<category>")[1].strip()
    return replace_category(result)

In [ ]:
print("正在测试集上评估使用思维链的RAG方法...")
evaluate(X_test, y_test, rag_chain_of_thought_classify)

# 评估

本指南说明了在提示工程中经验性测量提示性能的重要性。您可以在[此处](https://docs.anthropic.com/en/docs/prompt-engineering)阅读更多关于我们提示工程经验方法的内容。使用Jupyter Notebook是开始提示工程的好方法，但随着数据集变得更大、提示变得更多，重要的是要利用能够随您扩展的工具。

在本指南的这一部分，我们将探索使用[Promptfoo](https://www.promptfoo.dev/)，一个开源的LLM评估工具包。首先请前往`./evaluation`目录并查看`./evaluation/README.md`。

当您成功运行评估后，请回到这里查看结果。

In [ ]:
import json
import pandas as pd

promptfoo_results = pd.read_csv("./data/results.csv")
examples_columns = promptfoo_results.columns[2:]

number_of_providers = 5
number_of_prompts = 3

prompts = ["Simple", "RAG", "RAG w/ CoT"]

columns = ["label", "text"] + [
    json.loads(examples_columns[prompt * number_of_providers + provider])["provider"]
    + " Prompt: "
    + str(prompts[prompt])
    for prompt in range(number_of_prompts)
    for provider in range(number_of_providers)
]

promptfoo_results.columns = columns

result = (
    promptfoo_results.iloc[:, 2:]
    .astype(str)
    .apply(lambda x: x.str.count("PASS"))
    .sum()
    / len(promptfoo_results)
    * 100
).sort_values(ascending=False)

print(result)